# 12.4 - Caching Strategies

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook works through Caching Strategies hands-on: The economics: why cache at all; Exact-match cache: hash the normalized prompt; Semantic cache: catch paraphrases, mind the false positives; Provider prompt caching: cache the prefix, ~90 percent off input; TTL and eviction: bound the cache; Invalidation and staleness: the hard problem.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the packages this lesson can use, then run every cell keyless. The demos in this notebook all use toy stand-ins (a bag-of-words embedder, a tick clock, illustrative prices) so nothing here actually needs a network call.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" python-dotenv -q  # noqa


**Explanation**

One line of environment prep: install the Anthropic SDK and python-dotenv. The install line is commented out so the cell is a no-op on a machine that already has them.

**How the code works - step by step**
- **`pip install "anthropic>=0.40.0" python-dotenv`** - the Anthropic SDK (only needed for the illustrative Step 8 snippet) and dotenv for loading keys from a `.env` file. Commented out; uncomment on a fresh Colab/env.

**In one line:** optional install, nothing runs by default.

**What you'll see:** (no output - environment prep)

## Setup (continued) - load an API key from the environment

Read any provider key from the environment instead of hardcoding it. None of the runnable cells below actually use it - this is here so the one illustrative production snippet at the end has a key to reference if you ever un-comment it.

> **Optional Anthropic API key** (set ANTHROPIC_API_KEY) - only for the final illustrative snippet; every runnable cell is keyless.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com


**Explanation**

A single defensive line that ensures `ANTHROPIC_API_KEY` exists in the environment (as an empty string if unset) so later code can read it without a KeyError. It is setup, not a model call - no request is made.

**How the code works - step by step**
- **`os.environ.setdefault("ANTHROPIC_API_KEY", "")`** - registers the key name if it is not already set, without overwriting a real value you may have exported. The comment points you to console.anthropic.com to get one.

**In one line:** make sure the key variable exists; never hardcode the secret.

**What you'll see:** (no output - environment prep)

## 1 - The economics: why cache at all

Before building any cache, prove it pays. This cell is a spreadsheet in code: it models 1,000 requests at a few hit rates and shows how cost and latency fall in step with the hit rate. It also encodes the practitioner rules of thumb - the 25-45 percent target band and the 'above 60 percent you may not need an LLM' warning.

In [ ]:
# THE ECONOMICS - a cache turns a repeated LLM call into a near-free lookup. It is the single
# highest-leverage cost and latency win, WITHIN limits: aim for a 25-45% hit rate. Prices and
# latencies are illustrative; the arithmetic is real. No key.

REQUESTS = 1000
LLM_COST, CACHE_COST = 0.0020, 0.00001       # $ per call (illustrative)
LLM_MS,   CACHE_MS   = 800, 5                 # latency per call (illustrative)

def model(hit_rate):
    hits = round(REQUESTS * hit_rate)
    misses = REQUESTS - hits                   # only misses actually call the LLM
    cost = misses * LLM_COST + hits * CACHE_COST
    base_cost = REQUESTS * LLM_COST            # cost with NO cache
    avg_ms = (misses * LLM_MS + hits * CACHE_MS) / REQUESTS
    base_ms = LLM_MS
    return hits, cost, base_cost, avg_ms, base_ms

print("1,000 requests, cost/latency vs cache hit rate:")
for h in (0.0, 0.30, 0.45, 0.67):
    hits, cost, base, avg_ms, base_ms = model(h)
    print(f"  hit rate {h:>4.0%}: {hits:>3} served from cache -> ${cost:6.3f} "
          f"(vs ${base:.2f}, {1-cost/base:.0%} saved), avg {avg_ms:.0f}ms (vs {base_ms}ms)")

print("\nRules of thumb:")
print("  target 25-45% hit rate: real savings without over-caching")
print("  above ~60%: the workload is so repetitive you may not need an LLM at all")
print("  under ~1,000 requests/day: the caching engineering can cost more than it saves")
# Output:
# 1,000 requests, cost/latency vs cache hit rate:
#   hit rate   0%:   0 served from cache -> $ 2.000 (vs $2.00, 0% saved), avg 800ms (vs 800ms)
#   hit rate  30%: 300 served from cache -> $ 1.403 (vs $2.00, 30% saved), avg 562ms (vs 800ms)
#   hit rate  45%: 450 served from cache -> $ 1.105 (vs $2.00, 45% saved), avg 442ms (vs 800ms)
#   hit rate  67%: 670 served from cache -> $ 0.667 (vs $2.00, 67% saved), avg 267ms (vs 800ms)
#
# Rules of thumb:
#   target 25-45% hit rate: real savings without over-caching
#   above ~60%: the workload is so repetitive you may not need an LLM at all
#   under ~1,000 requests/day: the caching engineering can cost more than it saves

**Explanation**

A pure arithmetic harness - no model, no cache, just the economics. `model()` splits requests into hits and misses, charges only the misses the LLM price, and returns the blended cost and average latency so you can compare each hit rate against the no-cache baseline.

**How the code works - step by step**
- **Constants** - `LLM_COST`/`CACHE_COST` and `LLM_MS`/`CACHE_MS` set illustrative per-call price and latency; a cache call is ~200x cheaper and ~160x faster.
- **`model(hit_rate)`** - computes `hits` and `misses`, then `cost = misses*LLM + hits*CACHE` (only misses pay the LLM), the no-cache `base_cost`, and the blended average latency.
- **The loop** - runs 0%, 30%, 45%, 67% and prints each as savings versus the uncached bill.
- **Rules of thumb** - prints the 25-45% target, the >60% red flag, and the <1,000 req/day 'not worth it' floor.

**In one line:** only misses cost money, so savings scale directly with the hit rate.

**What you'll see:** A table of the four hit rates: at 30% you save 30% (~$1.40 vs $2.00, 562ms vs 800ms), at 45% nearly half, at 67% two-thirds - followed by the three rules-of-thumb lines.

## 2 - Exact-match cache: hash the normalized prompt

The first cache you always build. Normalize the prompt, hash it to a short key, store the answer under it, and look it up in O(1). It is completely safe - only genuinely identical text hits - but that safety is also its ceiling: every paraphrase misses.

In [ ]:
# EXACT-MATCH CACHE - hash the NORMALIZED prompt to a key, store the response, look it up in
# O(1). Dead simple and safe, but it only catches IDENTICAL requests - every paraphrase misses.
# No key.
import hashlib

def normalize(prompt):                         # lowercase, trim, collapse internal whitespace
    return " ".join(prompt.lower().strip().split())

def key(prompt):
    return hashlib.md5(normalize(prompt).encode()).hexdigest()[:10]

cache = {}
def ask(prompt, answer=None):
    k = key(prompt)
    if k in cache:
        return "HIT ", cache[k]
    cache[k] = answer or "(freshly generated)"
    return "MISS", cache[k]

print("Populate the cache, then look it up:")
print(f"  {ask('What is the refund policy?', 'Refunds within 30 days.')[0]}  key={key('What is the refund policy?')}  'What is the refund policy?'")
# An identical request (different case + spacing) normalizes to the SAME key -> HIT.
print(f"  {ask('  what is the REFUND policy?  ')[0]}  key={key('  what is the REFUND policy?  ')}  'what is the REFUND policy?' (same after normalize)")
# A paraphrase asks the same thing but hashes to a DIFFERENT key -> MISS.
print(f"  {ask('How do refunds work?')[0]}  key={key('How do refunds work?')}  'How do refunds work?' (a paraphrase)")
print("\nExact-match hits only identical text. Most repeated questions are worded differently -> Step 3.")
# Output:
# Populate the cache, then look it up:
#   MISS  key=f452fc0bc0  'What is the refund policy?'
#   HIT   key=f452fc0bc0  'what is the REFUND policy?' (same after normalize)
#   MISS  key=7c0c146037  'How do refunds work?' (a paraphrase)
#
# Exact-match hits only identical text. Most repeated questions are worded differently -> Step 3.

**Explanation**

A minimal cache-aside dictionary keyed by a hash of the normalized prompt. The point of the cell is to demonstrate the blind spot: normalization folds together case/whitespace variants, but a reworded question hashes somewhere else entirely.

**How the code works - step by step**
- **`normalize()`** - lowercases, strips, and collapses internal whitespace so trivially-different text maps to one string.
- **`key()`** - MD5-hashes the normalized text and takes the first 10 hex chars as a compact key.
- **`ask()`** - the cache-aside move: return `HIT` if the key is present, otherwise store the answer and return `MISS`.
- **The three calls** - populate with the canonical question (MISS), re-ask it with different case/spacing (same key -> HIT), then ask a paraphrase (different key -> MISS).

**In one line:** normalize -> hash -> lookup; identical text hits, paraphrases don't.

**What you'll see:** Three lines: the first ask MISSes (key f452fc0bc0), the re-cased/spaced repeat HITs on the same key, and the paraphrase 'How do refunds work?' MISSes on a new key (7c0c146037).

## 3 - Semantic cache: catch paraphrases, mind the false positives

To catch the paraphrases exact-match misses, cache by meaning: embed the query and return a stored answer if it is within a cosine-similarity threshold. This cell also shows the trap - a loose threshold serves a wrong-but-plausible answer, and the embedding model matters more than the threshold.

In [ ]:
# SEMANTIC CACHE - embed the query and return a cached answer if a stored query is within a
# cosine-similarity THRESHOLD. This catches paraphrases exact-match misses - but the FALSE
# POSITIVE is the whole game: too loose returns a wrong answer to a different question. A toy
# bag-of-words embedder stands in for a real model, so this runs with no key.
import math
STOP = {"what", "whats", "is", "the", "a", "how", "does", "do", "your", "are", "of"}

def embed(text):                               # toy: the set of content words (a real model is far better)
    return {w.strip("?.,") for w in text.lower().replace("'", "").split() if w.strip("?.,") not in STOP}

def cosine(a, b):
    if not a or not b: return 0.0
    return len(a & b) / (math.sqrt(len(a)) * math.sqrt(len(b)))

STORE = [("what is the refund policy", "Refunds are accepted within 30 days.")]
stored_vec = embed(STORE[0][0])

def semantic_lookup(query, threshold):
    v = embed(query)
    sim = cosine(v, stored_vec)
    return sim, ("HIT " if sim >= threshold else "MISS"), STORE[0][1]

queries = ["whats the refund policy?",      # a clean paraphrase (same intent)
           "how does a refund work?",        # a looser paraphrase (same intent)
           "what is the privacy policy?"]    # a DIFFERENT question that shares the word 'policy'
for thr in (0.45, 0.90):
    print(f"threshold {thr}:")
    for q in queries:
        sim, verdict, ans = semantic_lookup(q, thr)
        note = ""
        if verdict == "HIT " and q.startswith("what is the privacy"):
            note = "  <- FALSE HIT: returns the REFUND answer to a PRIVACY question (wrong!)"
        print(f"  sim={sim:.2f} {verdict} {q:32}{note}")
    print()
print("The weak embedder cannot separate 'refund work' from 'privacy policy' (both 0.50): loose")
print("catches the paraphrase but false-hits privacy; tight avoids it but misses the paraphrase.")
print("So the EMBEDDING MODEL matters most - a better one places intents where a threshold can split them.")
# Output:
# threshold 0.45:
#   sim=1.00 HIT  whats the refund policy?
#   sim=0.50 HIT  how does a refund work?
#   sim=0.50 HIT  what is the privacy policy?       <- FALSE HIT: returns the REFUND answer to a PRIVACY question (wrong!)
#
# threshold 0.9:
#   sim=1.00 HIT  whats the refund policy?
#   sim=0.50 MISS how does a refund work?
#   sim=0.50 MISS what is the privacy policy?
#
# The weak embedder cannot separate 'refund work' from 'privacy policy' (both 0.50): loose
# catches the paraphrase but false-hits privacy; tight avoids it but misses the paraphrase.
# So the EMBEDDING MODEL matters most - a better one places intents where a threshold can split them.

**Explanation**

A semantic cache built on a deliberately weak bag-of-words embedder so it runs with no key. Running the same three queries at a loose and a tight threshold makes the false-positive/missed-hit tradeoff visible, and shows why a poor embedder makes it unwinnable.

**How the code works - step by step**
- **`embed()`** - a toy embedding: the set of content words after dropping stopwords (a real model is far better).
- **`cosine()`** - similarity as the normalized overlap of two word-sets.
- **`STORE` / `stored_vec`** - one cached entry (the refund policy) and its precomputed vector.
- **`semantic_lookup()`** - embeds the query, scores it against the stored vector, and returns HIT/MISS versus the threshold.
- **The two threshold passes (0.45, 0.90)** - run a clean paraphrase, a looser paraphrase, and a different question ('privacy policy') that shares the word 'policy'; the code flags the dangerous false hit.

**In one line:** embed -> compare -> threshold decides; loose over-hits, tight under-hits, and the embedder sets the ceiling.

**What you'll see:** At threshold 0.45 the clean paraphrase (sim 1.00) and both 0.50 queries HIT - including a flagged FALSE HIT serving the refund answer to a privacy question; at 0.90 the false hit is gone but the real paraphrase now MISSes too.

## 4 - Provider prompt caching: cache the prefix, ~90 percent off input

A completely different cache from Steps 2-3: this one caches the *input* prefix, not the response. Send a long stable prefix (a big system prompt or document) and the provider bills its cached tokens at ~0.1x on reuse. This cell models the prefix-reuse economics.

In [ ]:
# PROVIDER PROMPT CACHING - a DIFFERENT cache: the provider caches a long, stable prompt PREFIX
# (a big system prompt or document) so the next call's cached INPUT tokens cost ~0.1x (a 90%
# discount). It caches input, not the response - orthogonal to Steps 2-3. Illustrative prices/tokens.

PREFIX_TOK, SUFFIX_TOK, CALLS = 10_000, 200, 5     # a 10k-token system prompt reused across 5 calls
PRICE = 3.0 / 1_000_000                             # $ per input token (illustrative)
WRITE_MULT, READ_MULT = 1.25, 0.1                   # cache write 1.25x, cache read 0.1x (Anthropic-style)

no_cache = CALLS * (PREFIX_TOK + SUFFIX_TOK) * PRICE        # every call pays full price for the whole prefix
# With prompt caching: call 1 WRITES the prefix (1.25x); calls 2..N READ it (0.1x). Suffix always full price.
write_cost = (PREFIX_TOK * WRITE_MULT + SUFFIX_TOK) * PRICE
read_cost  = (CALLS - 1) * (PREFIX_TOK * READ_MULT + SUFFIX_TOK) * PRICE
cached = write_cost + read_cost

print(f"A {PREFIX_TOK:,}-token prefix reused across {CALLS} calls (each + a {SUFFIX_TOK}-token suffix):")
print(f"  no prompt caching: every call pays full input price   -> ${no_cache:.4f}")
print(f"  with prompt caching: call 1 writes (1.25x), calls 2-5 read (0.1x) -> ${cached:.4f}")
print(f"  saved: ${no_cache - cached:.4f}  ({1 - cached/no_cache:.0%} off the input bill)")
print("\nKey: put the STABLE content first (it is cached by prefix); the cache matches up to the first change.")
print("Anthropic: manual cache_control + a write surcharge. OpenAI: automatic for prefixes >= 1,024 tokens.")
# Output:
# A 10,000-token prefix reused across 5 calls (each + a 200-token suffix):
#   no prompt caching: every call pays full input price   -> $0.1530
#   with prompt caching: call 1 writes (1.25x), calls 2-5 read (0.1x) -> $0.0525
#   saved: $0.1005  (66% off the input bill)
#
# Key: put the STABLE content first (it is cached by prefix); the cache matches up to the first change.
# Anthropic: manual cache_control + a write surcharge. OpenAI: automatic for prefixes >= 1,024 tokens.

**Explanation**

An economics model, not an API call. It compares paying full input price on every call against the provider-cache pattern where call 1 writes the prefix at a surcharge and later calls read it at a deep discount, holding the changing suffix at full price throughout.

**How the code works - step by step**
- **Constants** - a 10,000-token prefix reused across 5 calls, each with a 200-token suffix, at an illustrative per-token input price; `WRITE_MULT=1.25`, `READ_MULT=0.1` (Anthropic-style).
- **`no_cache`** - baseline: every call pays full price for prefix + suffix.
- **`write_cost`** - call 1 writes the prefix at 1.25x plus its full-price suffix.
- **`read_cost`** - calls 2-N read the prefix at 0.1x plus their full-price suffixes.
- **Print** - the two totals, the dollars saved, and the percentage off the input bill, plus the 'stable content first' rule and the Anthropic-manual / OpenAI-automatic ergonomics.

**In one line:** write the prefix once at 1.25x, reread it at 0.1x - a big recurring discount on tokens you were already sending.

**What you'll see:** The reused 10k-token prefix costs $0.1530 uncached vs $0.0525 with prompt caching - $0.1005 saved, 66% off the input bill - followed by the prefix-ordering and provider-ergonomics notes.

## 5 - TTL and eviction: bound the cache

A real cache cannot grow forever, so it needs two ways to let go: a TTL that expires entries by age, and an eviction policy that drops something when the cache is full. This cell implements an LRU cache with a TTL using a tick clock for time.

In [ ]:
# TTL & EVICTION - a cache is finite, so entries EXPIRE on a time-to-live and are EVICTED by a
# policy. LRU (least-recently-used) is the common default. A tick clock stands in for time. No key.
from collections import OrderedDict

class LRUCacheTTL:
    def __init__(self, capacity, ttl):
        self.cap, self.ttl, self.store = capacity, ttl, OrderedDict()   # key -> (value, inserted_tick)
    def get(self, key, now):
        if key not in self.store: return None
        value, ins = self.store[key]
        if now - ins > self.ttl:                                        # expired
            del self.store[key]; return "EXPIRED"
        self.store.move_to_end(key)                                     # mark most-recently-used
        return value
    def put(self, key, value, now):
        if key in self.store: self.store.move_to_end(key)
        self.store[key] = (value, now)
        if len(self.store) > self.cap:                                  # evict least-recently-used
            evicted, _ = self.store.popitem(last=False)
            return evicted
        return None

c = LRUCacheTTL(capacity=3, ttl=10)
print("LRU cache, capacity 3, TTL 10 ticks:")
for k in "ABC": c.put(k, f"ans_{k}", now=0)
print(f"  inserted A,B,C -> cache holds {list(c.store)}")
c.get("A", now=1)                                                       # A is now most-recently-used
print(f"  read A (A is now MRU)          -> order {list(c.store)}")
ev = c.put("D", "ans_D", now=2)
print(f"  insert D (over capacity)       -> evicted {ev} (the least-recently-used), holds {list(c.store)}")
print(f"  read C at tick 15 (> TTL 10)   -> {c.get('C', now=15)} (expired and dropped)")
print("\nTradeoff: a bigger cache lifts the hit rate but costs memory; a shorter TTL keeps answers fresh.")
# Output:
# LRU cache, capacity 3, TTL 10 ticks:
#   inserted A,B,C -> cache holds ['A', 'B', 'C']
#   read A (A is now MRU)          -> order ['B', 'C', 'A']
#   insert D (over capacity)       -> evicted B (the least-recently-used), holds ['C', 'A', 'D']
#   read C at tick 15 (> TTL 10)   -> EXPIRED (expired and dropped)
#
# Tradeoff: a bigger cache lifts the hit rate but costs memory; a shorter TTL keeps answers fresh.

**Explanation**

A small but real LRU+TTL cache built on `OrderedDict`. The ordering encodes recency (most-recently-used at the end), the stored insertion tick encodes age, and the demo walks through a read that promotes an entry, an insert that evicts the LRU, and a lookup that expires a stale entry.

**How the code works - step by step**
- **`__init__`** - holds `capacity`, `ttl`, and an `OrderedDict` mapping key -> (value, inserted_tick).
- **`get()`** - returns None if absent, `"EXPIRED"` (and deletes) if `now - inserted > ttl`, otherwise moves the key to the end (marks it most-recently-used) and returns the value.
- **`put()`** - inserts/updates with the current tick, then if over capacity pops the *first* item (the least-recently-used) and returns which key was evicted.
- **The walkthrough** - insert A,B,C; read A (promotes it); insert D (evicts B); read C past its TTL (EXPIRED).

**In one line:** recency order drives eviction, insertion tick drives expiry.

**What you'll see:** Cache holds [A,B,C]; reading A reorders to [B,C,A]; inserting D evicts B leaving [C,A,D]; reading C at tick 15 (> TTL 10) returns EXPIRED - then the capacity-vs-freshness tradeoff line.

## 6 - Invalidation and staleness: the hard problem

The genuinely hard part: a stale cached answer still looks valid, it is just wrong. The fix is to bake scope and version into the key so any change produces a new key. This cell contrasts a correctly versioned key with an un-versioned one that serves a dangerous stale hit.

In [ ]:
# INVALIDATION & STALENESS - the hard problem. A stale cached answer still LOOKS valid; it is
# just wrong. The fix: cache keys carry SCOPE (tenant/user) and VERSION (model, prompt, source), and
# an entry is invalid when any of those change. No key.
import hashlib

def versioned_key(prompt, model, prompt_ver, source_ver):
    raw = f"{prompt}|{model}|{prompt_ver}|{source_ver}"
    return hashlib.md5(raw.encode()).hexdigest()[:10]

cache = {}
def ask(prompt, model, prompt_ver, source_ver, answer):
    k = versioned_key(prompt, model, prompt_ver, source_ver)
    if k in cache: return "HIT ", cache[k]
    cache[k] = answer; return "MISS", cache[k]

print("A versioned, scoped cache key invalidates automatically when anything changes:")
r = ask("price?", "claude-sonnet-4-6", "v1", "prices-2026-06", "The course is 14999.")
print(f"  {r[0]} model=claude-sonnet-4-6 prompt=v1 source=prices-2026-06 -> {r[1]}")
r = ask("price?", "claude-sonnet-4-6", "v1", "prices-2026-06", "...")          # same versions
print(f"  {r[0]} identical versions -> cache HIT (correct)")
r = ask("price?", "claude-opus-4-8",   "v1", "prices-2026-06", "The course is 14999 (opus).")
print(f"  {r[0]} model upgraded -> DIFFERENT key, old entry not served (correctly invalidated)")

# The danger: an UN-versioned key that ignores the source version.
print("\nAn un-versioned key serves a STALE hit after the source changes:")
naive = {}
naive[hashlib.md5(b'price?').hexdigest()[:10]] = "The course is 14999."          # cached under the OLD price
# ... the price is updated to 19999, but the naive key is the same, so:
served = naive[hashlib.md5(b'price?').hexdigest()[:10]]
print(f"  price updated to 19999, but the un-versioned key returns: '{served}'  <- STALE: looks valid, is WRONG")
print("Rule: key on model + prompt + source version; and NEVER cache personalized or time-sensitive answers.")
# Output:
# A versioned, scoped cache key invalidates automatically when anything changes:
#   MISS model=claude-sonnet-4-6 prompt=v1 source=prices-2026-06 -> The course is 14999.
#   HIT  identical versions -> cache HIT (correct)
#   MISS model upgraded -> DIFFERENT key, old entry not served (correctly invalidated)
#
# An un-versioned key serves a STALE hit after the source changes:
#   price updated to 19999, but the un-versioned key returns: 'The course is 14999.'  <- STALE: looks valid, is WRONG
# Rule: key on model + prompt + source version; and NEVER cache personalized or time-sensitive answers.

**Explanation**

A side-by-side demonstration. The versioned cache mixes the prompt with the model, prompt-version, and source-version, so a model upgrade or data change automatically routes to a fresh key; the naive cache keys on prompt text alone and keeps serving the old answer after the underlying price changes.

**How the code works - step by step**
- **`versioned_key()`** - hashes `prompt|model|prompt_ver|source_ver` so every dimension is part of the identity.
- **`ask()`** - standard cache-aside over the versioned key (HIT if present, else store and MISS).
- **The versioned run** - MISS on first ask; HIT on an identical re-ask; MISS after upgrading the model (new key = automatic invalidation).
- **The naive run** - stores an answer under `md5('price?')`, 'updates' the real price, then reads the same un-versioned key back and gets the stale old price.

**In one line:** put model+prompt+source in the key, or a changed world keeps serving a plausible wrong answer.

**What you'll see:** The versioned cache MISS/HIT/MISS shows correct auto-invalidation on the model upgrade; the naive cache returns the old '14999' price after it was updated to 19999 - flagged STALE: looks valid, is WRONG.

## 7 - The multi-tier cache stack

Compose everything in cost order: check the exact-match cache first (O(1)), fall through to the semantic cache on a miss, and only on a full miss call the LLM - then populate both caches so the next repeat is cheap. Because it is now a system, the cell measures per-layer hits, the overall hit rate, and cost saved.

In [ ]:
# THE MULTI-TIER CACHE STACK - compose the layers in cost order: exact-match (O(1)) first, then a
# semantic check (catches paraphrases), and only on a full MISS does the real LLM call run and
# populate both caches. Track per-layer hits and the overall hit rate + cost saved. No key.
import math, hashlib
STOP = {"what", "whats", "is", "the", "a", "how", "does", "do", "your", "of"}
def embed(t): return {w.strip("?.,") for w in t.lower().replace("'","").split() if w.strip("?.,") not in STOP}
def cosine(a, b): return 0.0 if not a or not b else len(a & b)/(math.sqrt(len(a))*math.sqrt(len(b)))
def norm(p): return " ".join(p.lower().strip().split())

exact, semantic = {}, []           # semantic: list of (vec, answer)
THRESHOLD, LLM_COST = 0.8, 0.002
stats = {"exact": 0, "semantic": 0, "llm": 0, "cost": 0.0}

def answer(prompt):
    k = hashlib.md5(norm(prompt).encode()).hexdigest()[:8]
    if k in exact:                                  # tier 1: exact-match
        stats["exact"] += 1; return "exact", exact[k]
    v = embed(prompt)
    for vec, ans in semantic:                       # tier 2: semantic
        if cosine(v, vec) >= THRESHOLD:
            stats["semantic"] += 1; exact[k] = ans; return "semantic", ans
    resp = f"LLM answer to '{prompt[:24]}'"          # tier 3: MISS -> call the LLM, populate both
    stats["llm"] += 1; stats["cost"] += LLM_COST
    exact[k] = resp; semantic.append((v, resp)); return "llm (miss)", resp

log = ["What is the refund policy?",     # miss -> LLM
       "What is the refund policy?",     # exact hit
       "what is the refund policy",      # exact hit (normalized)
       "What's the refund policy?",      # semantic hit (paraphrase)
       "How do I reset my password?"]    # miss -> LLM
print("A request log through exact -> semantic -> LLM:")
for p in log:
    tier, _ = answer(p)
    print(f"  {tier:12} <- {p}")
served_from_cache = stats["exact"] + stats["semantic"]
print(f"\nlayer hits: exact={stats['exact']} semantic={stats['semantic']} llm-miss={stats['llm']}")
print(f"hit rate: {served_from_cache}/{len(log)} = {served_from_cache/len(log):.0%}  |  LLM cost ${stats['cost']:.3f} "
      f"(vs ${len(log)*LLM_COST:.3f} uncached, {1 - stats['cost']/(len(log)*LLM_COST):.0%} saved)")
# Output:
# A request log through exact -> semantic -> LLM:
#   llm (miss)   <- What is the refund policy?
#   exact        <- What is the refund policy?
#   semantic     <- what is the refund policy
#   semantic     <- What's the refund policy?
#   llm (miss)   <- How do I reset my password?
#
# layer hits: exact=1 semantic=2 llm-miss=2
# hit rate: 3/5 = 60%  |  LLM cost $0.004 (vs $0.010 uncached, 60% saved)

**Explanation**

The capstone that wires Steps 2 and 3 into one fall-through pipeline and instruments it. `answer()` runs the tiers cheapest-first and records where each request was served; the driver replays a small request log and reports the metrics that tell you the stack is working.

**How the code works - step by step**
- **Helpers** - `embed`/`cosine`/`norm` reuse the toy embedder and the normalizer from earlier steps.
- **State** - `exact` dict, `semantic` list of (vec, answer), a `THRESHOLD`, and a `stats` counter.
- **`answer()`** - tier 1: exact-match hash lookup; tier 2: scan the semantic store for cosine >= threshold (and back-fill the exact key on a semantic hit); tier 3: full miss calls the LLM, charges cost, and populates *both* caches.
- **The log** - five requests (a fresh question, an exact repeat, a normalized repeat, a paraphrase, a new question) print the serving tier for each.
- **Metrics** - sums exact+semantic as cache-served and prints per-layer hits, the overall hit rate, and the cost saved.

**In one line:** cheapest check first, brew only on a full miss, then measure every layer.

**What you'll see:** The log shows llm-miss, exact, semantic, semantic, llm-miss; the summary reports exact=1 semantic=2 llm-miss=2, a 3/5 = 60% hit rate, and $0.004 vs $0.010 uncached (60% saved).

## 8 - The production stack: GPTCache, Redis LangCache, and cache_control

You do not hand-roll the previous cells in production. This is an illustrative, commented-out map of the real tools: wrap the client with GPTCache or Redis LangCache for semantic caching, and add `cache_control` for provider prompt caching. Nothing here executes.

> **Reference only** - the code is commented out and needs `pip install gptcache anthropic` plus provider keys to run.

In [ ]:
# THE PRODUCTION STACK - a semantic cache + provider prompt caching (illustrative minimal example).
# You do not hand-roll these in production: wrap the client with GPTCache (or Redis LangCache) for
# semantic caching, and add cache_control for provider prompt caching.

# --- Semantic cache with GPTCache (wraps the client in ~2 lines) ---
# pip install gptcache
# from gptcache import cache
# from gptcache.adapter import openai              # a drop-in OpenAI adapter
# cache.init()                                      # embedding + vector store + similarity, handled for you
# cache.set_openai_key()
# Your chat-completions call now checks the semantic cache BEFORE hitting the model.

# --- Redis LangCache: a caching-optimized embedder + a distance threshold ---
#   from langcache import LangCache
#   lc = LangCache(redis_url=..., embedding="redis/langcache-embed-v1", distance_threshold=0.15)
#   hit = lc.search(query)            # returns a cached answer if it is within the threshold

# --- Provider prompt caching: cache a long stable PREFIX (Anthropic) ---
# from anthropic import Anthropic
# client = Anthropic()
# client.messages.create(
#     model="claude-sonnet-4-6",
#     system=[{"type": "text", "text": LONG_SYSTEM_PROMPT,
#              "cache_control": {"type": "ephemeral"}}],      # cache this prefix -> ~90% cheaper on reuse
#     messages=[{"role": "user", "content": user_question}])
# Output: (illustrative minimal example - needs `pip install gptcache anthropic` + provider keys.)
# GPTCache/LangCache do semantic caching (the response); cache_control caches the INPUT prefix (~90% off).

**Explanation**

A reference cell, not a runnable one - every line is a comment. It maps the concepts you built by hand onto the libraries that productize them: GPTCache/LangCache for the semantic response cache, and Anthropic `cache_control` for the input-prefix cache.

**How the code works - step by step**
- **GPTCache block** - `cache.init()` + the OpenAI adapter wrap the client so a semantic-cache check runs before every completion (Steps 2-3 as a library).
- **Redis LangCache block** - a caching-optimized embedder and a `distance_threshold`; `lc.search(query)` returns a cached answer within the threshold.
- **Anthropic `cache_control` block** - marks a long system-prompt prefix `ephemeral` so it is cached for ~90% cheaper reuse (Step 4).

**In one line:** GPTCache/LangCache cache the response, `cache_control` caches the input prefix - compose both in production.

**What you'll see:** No output - it is a commented reference. The trailing comment notes GPTCache/LangCache do semantic response caching while cache_control caches the input prefix (~90% off).

Six caches, one discipline. The economics cell sets the target (a 25-45 percent hit rate); the exact-match and semantic cells raise it by widening what counts as a repeat; provider prompt caching discounts the reused input prefix orthogonally; TTL/LRU bounds the store; versioned keys keep it correct; and the final stack composes them in cost order and measures the result. Everything runs keyless with toy stand-ins, so you can feel each tradeoff before wiring the real backends. Next up: Lesson 12.7 scales the cache backend (Redis/Memorystore) and 12.8 monitors hit rate, false-positive rate, and cost savings.